In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from thesis.data_classes import AutoEncoderConfig
from thesis.csi_autoencoder import MultiCityGenerator, CSIAutoEncoder, train_model, plot_training_history

config = AutoEncoderConfig(latent_dim=128)
    
# 1. Load Data
train_gen = MultiCityGenerator(config.train_cities, config.scale_factor)
val_gen = MultiCityGenerator(config.val_cities, config.scale_factor)

X_train, _ = train_gen.load_all()
X_val, _ = val_gen.load_all()

train_dl = DataLoader(TensorDataset(X_train), batch_size=config.batch_size, shuffle=True)
val_dl = DataLoader(TensorDataset(X_val), batch_size=config.batch_size, shuffle=False)

# 2. Physics Calibration (Global Reference Power)
# Calculate average power of the training set to set the Noise Floor baseline
ref_power = torch.mean(torch.abs(X_train)**2).item()

# 3. Model
model = CSIAutoEncoder(latent_dim=config.latent_dim, mode="train")

# 4. Train
best_model, history, res_folder = train_model(model, train_dl, val_dl, config, ref_power)

# 5. Visualize
plot_training_history(history, save_dir=res_folder)